<a href="https://colab.research.google.com/github/narpavi-ai/lasp-agent-workshop/blob/main/notebooks/03-skill.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3 · A skill

**LASP workshop: build your agent**

Lab 2's agent got real numbers, but nobody told it how to do the job
*well*. What should it do when fits are tied? Which warnings must it
always report?

A **skill** is a written procedure the agent follows. Here it ships with
the tools, on the MCP server. About **10 requests**.

## Before you start

Add these with the **🔑 key icon** in Colab's left sidebar, and turn **Notebook access** on for each. Notebook access is per notebook, so each lab asks again.

| Name | Value |
|---|---|
| `GOOGLE_API_KEY` | your free key from <https://aistudio.google.com/apikey> |
| `LASP_MCP_TOKEN` | the workshop token, shown on the slide |
| `ANTHROPIC_API_KEY` | *optional:* only if you switch to Claude (a paid key from <https://console.anthropic.com>) |

## Setup

Run these in order. Each takes a few seconds.

### Setup · Install

LangChain talks to the model, LangGraph runs agents, and the MCP adapter connects to the LASP server. Exact versions, as tested, plus this workshop's helper file.

In [ ]:
# pip may print a warning about google-auth and google-colab. That's expected
# and harmless: Gemini's library needs a newer google-auth than Colab ships.
%pip install -q langchain==1.4.7 langchain-core==1.7.0 langchain-google-genai==4.4.2 langchain-anthropic==1.7.6 langgraph==1.2.17 langchain-mcp-adapters==0.3.2 mcp==1.30.0
!curl -sSfO https://raw.githubusercontent.com/narpavi-ai/lasp-agent-workshop/main/notebooks/lasp_lab.py
print("✅ Installed, and helpers downloaded.")

### Setup · Helpers

Display and connection helpers, so the notebook shows only what's being taught. Curious? Read [`lasp_lab.py`](lasp_lab.py).

In [ ]:
from lasp_lab import *   # show(), call(), run_live() and friends: see lasp_lab.py
print("✅ Helpers ready.")

### Setup · Choose your model

`"gemini"` is free and needs `GOOGLE_API_KEY`. `"claude"` is paid and needs `ANTHROPIC_API_KEY`. Nothing else in the lab changes.

In [ ]:
PROVIDER = "gemini"    # "gemini" or "claude"

### Setup · Load your key

Read from Colab Secrets, so it never appears in the notebook.

In [ ]:
import os
KEY_NAME = {"gemini": "GOOGLE_API_KEY", "claude": "ANTHROPIC_API_KEY"}[PROVIDER]
os.environ[KEY_NAME] = secret(KEY_NAME)
print(f"✅ {KEY_NAME} loaded.")

### Setup · Create the model

One line picks the provider and model. Everything after it is the same for both.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.rate_limiters import InMemoryRateLimiter

MODELS = {"gemini": "google_genai:gemini-3.5-flash-lite", "claude": "anthropic:claude-haiku-5-5"}
MODEL = MODELS[PROVIDER]

# Gemini's free tier allows 15 requests a minute; stay under it. Paid Claude keys allow more.
limiter = InMemoryRateLimiter(requests_per_second=0.2) if PROVIDER == "gemini" else None

# Claude 5 thinks before answering. Low effort keeps that short (faster, cheaper),
# and the cap leaves room for the answer after the thinking.
extra = {"max_tokens": 16000, "thinking": {"type": "adaptive"}, "output_config": {"effort": "low"}} if PROVIDER == "claude" else {}
model = init_chat_model(MODEL, rate_limiter=limiter, **extra)
print(f"✅ Model ready: {MODEL}")

### Setup · Connect to the LASP server

The same connection as Lab 2: the server's address, and the workshop token.

In [ ]:
LASP_MCP_URL = "https://lasp-mcp-189650845754.northamerica-northeast1.run.app/mcp"   # where the LASP server lives
LASP_MCP_TOKEN = secret("LASP_MCP_TOKEN")   # the workshop token, from Colab Secrets

client, tools = await connect(LASP_MCP_URL, LASP_MCP_TOKEN)
print(f"✅ Connected to the LASP server: {len(tools)} tools")

### Step 1 · The skill

Besides tools, an MCP server can offer **prompts**: reusable instructions
written by whoever knows the domain. Read the one the LASP server ships:

In [ ]:
skill = (await client.get_prompt("lasp", "adsorbent_screening",
                                 arguments={"gas1": "CO2", "gas2": "N2", "y1": "0.15", "pressure_bar": "1"}))[0].content
show_text(skill, "adsorbent_screening", icon="📋")

It has three parts, and each fixes something you saw in Lab 2:

- **A call plan:** which tools, how many times. No re-fitting the same data.
- **A rule for choosing:** the same rule for any data, not an answer for
  this data. It never says "use Toth"; it says how to break a tie.
- **An output format:** every answer has the same shape, so you (and
  Lab 4's judge) can compare runs.

That's a **skill**: a procedure, not code. The same idea appears as Agent
Skills (a folder with a `SKILL.md`) in Claude and other agent tools. Here
it travels with the server, so whoever maintains the tools maintains the
know-how too.

### Step 2 · Two agents

Same model, same tools. The only difference is the system prompt: one has
the skill appended.

In [ ]:
from langchain.agents import create_agent

agent_tools = [t for t in tools if not t.name.startswith("plot_")]
SYSTEM_PROMPT = ("You are a research assistant at the Laboratory for Advanced Separation Processes (LASP), "
         "University of Alberta, with the LASP Isotherm Fitter and IAST tools. "
         "Never estimate a number a tool can compute. Report the models you used and any "
         "workshop_validation warnings.")

plain = create_agent(model, agent_tools, system_prompt=SYSTEM_PROMPT)
skilled = create_agent(model, agent_tools, system_prompt=SYSTEM_PROMPT + "\n\nFollow this procedure:\n" + skill)
print("✅ Two agents: plain, and skilled.")

### Step 3 · Same question, both agents

This time the agent fetches the data itself from the server. It's **the
same data as Lab 2**: the bundled datasets hold exactly the numbers Lab 2
pasted into the prompt.

In [ ]:
USER_PROMPT = "Screen the bundled workshop_co2_298K_synthetic and workshop_n2_298K_synthetic datasets for CO2/N2 separation at 15% CO2 and 1 bar."

plain_messages = await run_live(plain, USER_PROMPT)

In [ ]:
skilled_messages = await run_live(skilled, USER_PROMPT)

### Compare

- **Tool calls:** count them in each trace. Did the skilled agent follow
  the call plan?
- **The choice:** Toth, Quadratic and Langmuir tie for N₂. The rule picks
  Langmuir (fewest parameters) and reports the range, S ≈ 388–420. What
  did the plain agent pick, and did it say why?
- **The format:** is the skilled answer in the table-and-lines format?
  Run it again: is the shape the same?
- Did it **report the extrapolation warning** with numbers?

"It looks better" isn't evidence, though. One run is one sample, and you
compared by eye. **Lab 4 measures it.**

## If something breaks

| What you see | Fix |
|---|---|
| `404 NOT_FOUND`, model not available | The provider retired the model. Pick a current one (<https://aistudio.google.com> or <https://docs.anthropic.com/en/docs/about-claude/models>) and change it in `MODELS`. |
| `401`, `API key not valid`, `authentication_error` | Check the key's secret (`GOOGLE_API_KEY` or `ANTHROPIC_API_KEY`): exact name, Notebook access on, and that `PROVIDER` matches it. |
| `Couldn't connect to …` | The message says why. A `401` means the `LASP_MCP_TOKEN` secret is wrong: 48 characters, `0-9` and `a-f` (a trailing `%` copied from the terminal isn't part of it). |
| `429`, `RESOURCE_EXHAUSTED` | Free-tier limit. Wait a minute and re-run the cell. |
| `ModuleNotFoundError`, or a name not defined | The runtime restarted. Run the setup cells again from the top. |